In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import json
import itertools

# load all data

data = {}

concurrency_counts = ['0', '1', '2']
time_lengths = ['500', '750', '1000']
browsers = ['operagx', 'chrome']
browsers_pretty = ['Opera GX', 'Google Chrome']
adblock_enabled = ['false', 'true']

for count, time, browser, adblocker in itertools.product(concurrency_counts, time_lengths, browsers, adblock_enabled):
    filepath = f'data/{count}con-{time}ms-data/{browser}-{adblocker}-times.json'
    with open(filepath, 'r') as file:
        file_data = np.mean(np.array(list(json.load(file).values())), axis=1)
    data[(count, time, browser, adblocker)] = file_data

In [ ]:
# Table 1
# Due to physical inspection of figures 1a, 1b, 1c, a cutoff of 29000 was used for timeout
cutoff = 29000

timeouts = []
for adblocker, time in itertools.product(adblock_enabled, time_lengths):
    print(f'{adblocker} + {time}')
    timeouts.append([(data[(count, time, browser, adblocker)] > cutoff).sum() for browser, count in itertools.product(browsers, concurrency_counts)])
timeouts = np.array(timeouts)
print(timeouts)

for browser, count in itertools.product(browsers, concurrency_counts):
    print(f'{browser} + {count}')

fig, ax = plt.subplots()
fig.set_size_inches((6.5, 6.5))
ax.axis('off')
table = ax.table(cellText=timeouts, loc='center')
table.set_fontsize(10)
table.scale(1.5, 1.5)

plt.show()

In [ ]:
# 500ms graph, figure 1a
mkrs = {'true':'+', 'false':'.'}
colors = {'0':'cornflowerblue', '1':'g', '2':'r'}
plt.figure(figsize=((6.5, 4)))
time = time_lengths[0]
plt.title(f'CDF of Time Until Network Idle\n(idleTime: {time}ms)')
for count, adblocker in itertools.product(concurrency_counts, adblock_enabled):
    arr = np.mean(np.array(list(data[(count, time, browser, adblocker)] for browser in browsers)), axis=0)
    sorted_data = np.sort(arr)
    n = len(arr)
    y = np.arange(1, n + 1) / n
    plt.plot(sorted_data, y, marker=mkrs[adblocker], linestyle='-', color=colors[count], label=f'Concurrency: {count}, Ads: {"Blocked" if adblocker == "true" else "Allowed"}')
plt.grid(True, alpha=0.3)
plt.ylabel(f'Cumulative Probability')
plt.xlim(0,35000)
plt.ylim(0,1)

# 4. Add labels, title, legend, and grid
plt.xlabel("Time Until Network Idle (ms)")
plt.legend()  # Add a legend to distinguish the lines
plt.savefig(f'cdf-merged_browsers_{time}ms.png')
plt.show()

In [ ]:
# 750ms graph, figure 1b
mkrs = {'true':'+', 'false':'.'}
colors = {'0':'cornflowerblue', '1':'g', '2':'r'}
plt.figure(figsize=((6.5, 4)))
time = time_lengths[1]
plt.title(f'CDF of Time Until Network Idle\n(idleTime: {time}ms)')
for count, adblocker in itertools.product(concurrency_counts, adblock_enabled):
    arr = np.mean(np.array(list(data[(count, time, browser, adblocker)] for browser in browsers)), axis=0)
    sorted_data = np.sort(arr)
    n = len(arr)
    y = np.arange(1, n + 1) / n
    plt.plot(sorted_data, y, marker=mkrs[adblocker], linestyle='-', color=colors[count], label=f'Concurrency: {count}, Ads: {"Blocked" if adblocker == "true" else "Allowed"}')
plt.grid(True, alpha=0.3)
plt.ylabel(f'Cumulative Probability')
plt.xlim(0,35000)
plt.ylim(0,1)

# 4. Add labels, title, legend, and grid
plt.xlabel("Time Until Network Idle (ms)")
plt.legend()  # Add a legend to distinguish the lines
plt.savefig(f'cdf-merged_browsers_{time}ms.png')
plt.show()

In [ ]:
# 1000ms graph, figure 1c
mkrs = {'true':'+', 'false':'.'}
colors = {'0':'cornflowerblue', '1':'g', '2':'r'}
plt.figure(figsize=((6.5, 4)))
time = time_lengths[2]
plt.title(f'CDF of Time Until Network Idle\n(idleTime: {time}ms)')
for count, adblocker in itertools.product(concurrency_counts, adblock_enabled):
    arr = np.mean(np.array(list(data[(count, time, browser, adblocker)] for browser in browsers)), axis=0)
    sorted_data = np.sort(arr)
    n = len(arr)
    y = np.arange(1, n + 1) / n
    plt.plot(sorted_data, y, marker=mkrs[adblocker], linestyle='-', color=colors[count], label=f'Concurrency: {count}, Ads: {"Blocked" if adblocker == "true" else "Allowed"}')
plt.grid(True, alpha=0.3)
plt.ylabel(f'Cumulative Probability')
plt.xlim(0,35000)
plt.ylim(0,1)

# 4. Add labels, title, legend, and grid
plt.xlabel("Time Until Network Idle (ms)")
plt.legend()  # Add a legend to distinguish the lines
plt.savefig(f'cdf-merged_browsers_{time}ms.png')
plt.show()

In [ ]:
# Box and whisker plot, figure 2
fig, axs = plt.subplots(ncols=3, sharex=True, sharey=True)
fig.set_size_inches((6.5, 4))
ticks = browsers_pretty

def set_box_color(bp, color):
    plt.setp(bp['boxes'], color=color)
    plt.setp(bp['whiskers'], color=color)
    plt.setp(bp['caps'], color=color)
    plt.setp(bp['medians'], color=color)

for x in range(len(concurrency_counts)):
    plot = axs[x]
    count = concurrency_counts[x]
    ads = [np.mean(np.array(list(data[(count, time, browser, 'false')] for time in time_lengths))/1000.0, axis=0) for browser in browsers]
    no_ads = [np.mean(np.array(list(data[(count, time, browser, 'true')] for time in time_lengths))/1000.0, axis=0) for browser in browsers]
    
    bpl = plot.boxplot(no_ads, sym='', positions=np.array(range(len(no_ads)))*2.0-0.4, widths=0.6)
    bpr = plot.boxplot(ads, sym='', positions=np.array(range(len(ads)))*2.0+0.4, widths=0.6)
    set_box_color(bpr, '#D7191C') # colors are from http://colorbrewer2.org/
    set_box_color(bpl, '#2C7BB6')
    plot.set_xticks(range(0, len(ticks) * 2, 2), ticks)
    plot.set_xticklabels(ticks, rotation=15)
    plot.set_xlabel(f'Concurrency: {count}')
    plot.grid(True, axis='y', alpha=0.3)

    if x == 0:
        plot.set_ylabel('Time Until Network Idle (sec)')
    if x == 1:
        plot.set_title('Box Plots of Time Until Network Idle by Browser and Concurrency')
    if x == 2:
        plot.plot([], c='#D7191C', label='Ads')
        plot.plot([], c='#2C7BB6', label='No Ads')
        plot.legend()
plt.ylim(bottom=0)
plt.tight_layout()
plt.savefig('boxcompare.png')

In [ ]:
# Unused figure, not enough granularity in the figure
plt.figure(figsize=(10,6))
mkrs = {'true':'+', 'false':'.'}
colors = {'0':'cornflowerblue', '1':'g', '2':'r'}
for count, adblocker in itertools.product(concurrency_counts, adblock_enabled):
    arr = np.mean(np.array(list(data[(count, time, browser, adblocker)] for time, browser in itertools.product(time_lengths, browsers))), axis=0)
    sorted_data = np.sort(arr)
    n = len(arr)
    y = np.arange(1, n + 1) / n
    plt.plot(sorted_data, y, marker=mkrs[adblocker], linestyle='-', color=colors[count], label=f'{count}con-{"no_ads" if adblocker == "true" else "ads"}')

# 4. Add labels, title, legend, and grid
plt.title("CDF of Time Until Network Idle")
plt.xlabel("Time Until Network Idle (ms)")
plt.xlim(0,35000)
plt.ylabel("Cumulative Probability")
plt.ylim(0,1)
plt.grid(True, alpha=0.3)
plt.legend()  # Add a legend to distinguish the lines
plt.savefig('cdf-merge_browsers_and_times.png')
plt.show()

In [ ]:
# Unused figure, instead was split into cells above as figures 1a, 1b, & 1c
fig, axs = plt.subplots(nrows=3, sharex=False, sharey=True)
fig.set_size_inches((10, 7))
mkrs = {'true':'+', 'false':'.'}
colors = {'0':'cornflowerblue', '1':'g', '2':'r'}

for ypos in range(len(time_lengths)):
    plot = axs[ypos]
    time = time_lengths[ypos]
    if ypos == 0:
        plot.set_title('CDF of Time Until Network Idle')
    for count, adblocker in itertools.product(concurrency_counts, adblock_enabled):
        arr = np.mean(np.array(list(data[(count, time, browser, adblocker)] for browser in browsers)), axis=0)
        sorted_data = np.sort(arr)
        n = len(arr)
        y = np.arange(1, n + 1) / n
        plot.plot(sorted_data, y, marker=mkrs[adblocker], linestyle='-', color=colors[count], label=f'Concurrency: {count}, Ads: {"Blocked" if adblocker == "true" else "Allowed"}')
    plot.grid(True, alpha=0.3)
    plot.set_ylabel(f'Cumulative Probability\n(idleTime: {time}ms)')
    plot.set_xlim(0,35000)
    plot.set_ylim(0,1)

# 4. Add labels, title, legend, and grid
plt.xlabel("Time Until Network Idle (ms)")
plt.legend()  # Add a legend to distinguish the lines
plt.savefig('cdf-merged_browsers.png')
plt.show()

In [ ]:
# Unused figure, though very interesting in what it shows
fig, axs = plt.subplots(ncols=2, sharex=True, sharey=True)
fig.set_size_inches((16,4.8))
mkrs = {'true':'+', 'false':'.'}
colors = {'0':'cornflowerblue', '1':'g', '2':'r'}

for xpos in range(len(browsers)):
    plot = axs[xpos]
    browser = browsers[xpos]
    for count, adblocker in itertools.product(concurrency_counts, adblock_enabled):
        arr = np.mean(np.array(list(data[(count, time, browser, adblocker)] for time in time_lengths)), axis=0)
        sorted_data = np.sort(arr)
        n = len(arr)
        y = np.arange(1, n + 1) / n
        plot.plot(sorted_data, y, marker=mkrs[adblocker], linestyle='-', color=colors[count], label=f'{count}con-{"no_ads" if adblocker == "true" else "ads"}')
    plot.grid(True, alpha=0.3)
    plot.set_title(f'CDF of Time Until Network Idle ({"Opera GX" if browser == "operagx" else "Google Chrome"})')
    plot.set_ylabel('Cumulative Probability')

# 4. Add labels, title, legend, and grid
plt.xlim(0,35000)
plt.xlabel("Time Until Network Idle (ms)")
plt.ylim(0,1)
plt.legend()  # Add a legend to distinguish the lines
plt.savefig('cdf-merged_times.png')
plt.show()

In [ ]:
# Unused figure, too much space required
fig, axs = plt.subplots(3,2, sharex=True, sharey=True)
fig.set_size_inches((14, 12))
mkrs = {'true':'+', 'false':'.'}
colors = {'0':'cornflowerblue', '1':'g', '2':'r'}

for xpos in range(len(browsers)):
    browser = browsers[xpos]
    for ypos in range(len(time_lengths)):
        plot = axs[ypos,xpos]
        time = time_lengths[ypos]
        for count, adblocker in itertools.product(concurrency_counts, adblock_enabled):
            arr = data[(count, time, browser, adblocker)]
            sorted_data = np.sort(arr)
            n = len(arr)
            y = np.arange(1, n + 1) / n
            plot.plot(sorted_data, y, marker=mkrs[adblocker], linestyle='-', color=colors[count], label=f'Concurrency: {count}, Ads: {"Blocked" if adblocker == "true" else "Allowed"}')
        plot.grid(True, alpha=0.3)
        plot.set_title(f'CDF of Time Until Network Idle ({time}ms)')
        if (xpos == 0):
            plot.set_ylabel('Cumulative Probability')
    plot.set_xlabel(f'Time Until Network Idle (ms) ({"Opera GX" if browser == "operagx" else "Google Chrome"})')

# 4. Add labels, title, legend, and grid
plt.xlim(0,35000)
plt.ylim(0,1)
plt.legend()  # Add a legend to distinguish the lines
plt.savefig('cdf.png')
plt.tight_layout()
plt.show()